# ML Team Pipeline

This notebook contains the complete machine learning pipeline, including data processing, model training, and inference.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle
from scipy.optimize import linprog


## 1. Data Processing

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """Loads data from a given filepath."""
    print(f"Loading data from {filepath}...")
    return pd.read_csv(filepath)

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Performs basic data cleaning."""
    print("Cleaning data...")
    # Drop missing values for now
    return df.dropna()

def preprocess_features(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocesses features for modeling."""
    print("Preprocessing features...")
    # Add feature engineering steps here
    return df

## 2. Model Training

In [ ]:
def train_model(X: pd.DataFrame, y: pd.Series):
    """Trains a basic random forest classifier."""
    print("Splitting dataset...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print("Training model...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    print("Evaluating model...")
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Model Accuracy: {acc:.4f}")
    
    return model

## 3. Inference

In [ ]:
def save_model(model, filepath: str):
    """Saves a trained model to disk."""
    print(f"Saving model to {filepath}...")
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)

def load_model(filepath: str):
    """Loads a trained model from disk."""
    print(f"Loading model from {filepath}...")
    with open(filepath, 'rb') as f:
        return pickle.load(f)

def run_inference(model, new_data: pd.DataFrame) -> np.ndarray:
    """Runs inference on new data."""
    print("Running inference...")
    return model.predict(new_data)

## 4. Optimization Engine (SciPy Solver)

In [ ]:
def optimize_shipment(df: pd.DataFrame, total_budget: float):
    """
    Uses SciPy linprog to find the optimal shipping quantities
    to minimize costs while respecting capacity and budget constraints.
    """
    print("Setting up optimization problem...")
    
    # Objective: Minimize cost = sum(cost_i * x_i)
    c = df['Shipping_Cost_Per_Unit'].values
    
    # Constraint 1: Capacity bound
    bounds = [(0, cap) for cap in df['Capacity']]
    
    # Constraint 2: Budget (sum(cost_i * x_i) <= total_budget)
    A_ub = [c]
    b_ub = [total_budget]
    
    # Constraint 3: Demand (sum(x_i) >= total_demand) 
    total_demand = df['Demand'].sum()
    A_ub.append([-1] * len(df)) # -sum(x_i) <= -total_demand
    b_ub.append(-total_demand)
    
    print("Running solver...")
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    
    if result.success:
        print("Optimization successful!")
        return result.x
    else:
        print("Optimization failed:", result.message)
        return None

def generate_optimal_choices(df: pd.DataFrame):
    """Generates 3 choices based on different budget constraints."""
    choices = []
    # Base budget to fulfill demand = sum(min_cost_needed)
    budgets = [50000, 30000, 25000] # A: High, B: Medium, C: Low
    labels = ['Choice A (High Budget)', 'Choice B (Medium Budget)', 'Choice C (Low Budget)']
    
    for label, budget in zip(labels, budgets):
        print(f"\n--- {label} ---")
        sol = optimize_shipment(df, budget)
        if sol is not None:
            choices.append((label, sol))
            print(f"Optimal allocations: {np.round(sol, 2)}")
    return choices
